# Negotiating the Past - Sample Classification & Visualization

Test the MLX classification on a sample, then visualize and cluster the results.

## 1. Setup & Dependencies

In [ ]:
# Install dependencies -- uncomment to install
#%pip install mlx-lm pandas tqdm huggingface_hub
#%pip install --upgrade typing_extensions bertopic sentence-transformers umap-learn hdbscan plotly

In [ ]:
import os
import time
import logging
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Create directories
os.makedirs("data/results_mlx", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"logs/mlx_sample_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("mlx_classifier")
print("Libraries loaded successfully")

## 2. Load Model

In [ ]:
# Configuration
MODEL_ID = "mlx-community/Ministral-3-3B-Instruct-2512-4bit"

from mlx_lm import load, generate, batch_generate
from huggingface_hub import snapshot_download

print(f"Model: {MODEL_ID}")
print("=" * 60)

print("\nStep 1/2: Downloading model (if not cached)...")
local_path = snapshot_download(repo_id=MODEL_ID)
print(f"Model cached at: {local_path}")

print("\nStep 2/2: Loading model into memory...")
model, tokenizer = load(local_path)
print("=" * 60)
print("Model loaded successfully via MLX!")

## 3. System Prompt

In [ ]:
SYSTEM_PROMPT = """Analyze if this image generation prompt contains a reference to the historical past.

Answer YES if the prompt contains:
- Historical figures (Napoleon, Caesar, Cleopatra, Marie Antoinette, etc.)
- Historical events (World War, Revolution, Cold War, etc.)
- Historical periods or eras (Victorian, Medieval, Renaissance, 1920s, Ancient Rome, etc.)
- Historical artists or their works (Da Vinci, Rembrandt, Michelangelo, etc.)
- Historical art movements (Baroque, Art Nouveau, Impressionism, etc.)
- Mythology and ancient legends (Greek gods, Norse mythology, Egyptian mythology, etc.)

Answer NO if the prompt:
- Only uses stylistic words (vintage, retro, sepia, old photograph)
- Only describes fictional/fantasy content (steampunk, cyberpunk, sci-fi)
- Only mentions living celebrities in modern context
- References extinct animals without historical context (dinosaurs, dodo)
- Is purely futuristic with no past reference

Format: [yes/no]: [one sentence reason]
"""

print("System prompt configured")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

## 4. Classification Functions

In [ ]:
from mlx_lm.sample_utils import make_sampler

greedy_sampler = make_sampler(temp=0.0)

def format_prompt_for_model(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this prompt: {user_prompt}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def parse_response(response: str) -> tuple[str, str]:
    response = response.strip()
    match = re.match(r'^(yes|no)\s*[:\-]\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    if match:
        classification = match.group(1).lower()
        justification = match.group(2).strip()
        justification = justification.split('.')[0] + '.' if '.' in justification else justification
        return classification, justification
    response_lower = response.lower()
    if response_lower.startswith('yes'):
        return 'yes', response[3:].strip(' :-')
    elif response_lower.startswith('no'):
        return 'no', response[2:].strip(' :-')
    if 'yes' in response_lower[:50]:
        return 'yes', response
    elif 'no' in response_lower[:50]:
        return 'no', response
    logger.warning(f"Could not parse response: {response[:100]}")
    return 'error', response


def classify_prompt(prompt: str, max_tokens: int = 100) -> tuple[str, str, float]:
    formatted = format_prompt_for_model(prompt)
    start_time = time.time()
    response = generate(model, tokenizer, prompt=formatted, max_tokens=max_tokens, sampler=greedy_sampler)
    generation_time = time.time() - start_time
    classification, justification = parse_response(response)
    return classification, justification, generation_time


def classify_batch(prompts: list[str], max_tokens: int = 60) -> list[dict]:
    formatted_prompts = [format_prompt_for_model(p) for p in prompts]
    tokenized = [tokenizer.encode(p) for p in formatted_prompts]
    start_time = time.time()
    batch_response = batch_generate(model, tokenizer, prompts=tokenized, max_tokens=max_tokens)
    total_time = time.time() - start_time
    time_per_prompt = total_time / len(prompts)
    results = []
    for prompt, response_text in zip(prompts, batch_response.texts):
        classification, justification = parse_response(response_text)
        results.append({
            'prompt': prompt,
            'references_past': classification,
            'justification': justification,
            'generation_time': time_per_prompt
        })
    return results, total_time


BATCH_SIZE_INFERENCE = 8
print("Classification functions defined (with batch support)")

## 5. Load Sample Dataset

In [ ]:
SAMPLE_SIZE = 100

print(f"Loading {SAMPLE_SIZE} prompts for validation (with metadata)...")

total_rows = sum(1 for _ in open('data/prompts.csv')) - 1
print(f"Total prompts in dataset: {total_rows:,}")

skip_prob = 1 - SAMPLE_SIZE / total_rows
np.random.seed(42)
prompts_df = pd.read_csv(
    'data/prompts.csv',
    skiprows=lambda x: x > 0 and np.random.random() < skip_prob,
    nrows=SAMPLE_SIZE
)

def extract_metadata(raw_data_str):
    try:
        data = json.loads(raw_data_str)
        discord_data = data.get('raw_discord_data', {})
        ts = discord_data.get('timestamp', 0)
        if isinstance(ts, str):
            try:
                ts = pd.Timestamp(ts).timestamp()
            except:
                ts = 0
        return pd.Series({
            'image_uri': discord_data.get('image_uri', ''),
            'timestamp': float(ts) if ts else 0,
            'modifiers': json.dumps(data.get('modifiers', [])),
            'width': discord_data.get('width', 0),
            'height': discord_data.get('height', 0),
        })
    except (json.JSONDecodeError, TypeError):
        return pd.Series({'image_uri': '', 'timestamp': 0, 'modifiers': '[]', 'width': 0, 'height': 0})

print("Parsing metadata...")
metadata_cols = prompts_df['raw_data'].apply(extract_metadata)
prompts_df = pd.concat([prompts_df[['prompt']], metadata_cols], axis=1)
prompts_df['datetime'] = pd.to_datetime(prompts_df['timestamp'], unit='s', errors='coerce')
prompts_df.loc[prompts_df['timestamp'] == 0, 'datetime'] = pd.NaT
prompts_df = prompts_df.dropna(subset=['prompt'])
prompts_df['prompt'] = prompts_df['prompt'].astype(str).str.strip()
prompts_df = prompts_df[prompts_df['prompt'] != '']
prompts_list = prompts_df['prompt'].tolist()

print(f"Loaded {len(prompts_df)} prompts with metadata")
print(f"Date range: {prompts_df['datetime'].min()} to {prompts_df['datetime'].max()}")

## 6. Test Single Classification

In [ ]:
test_prompt = prompts_list[0] if prompts_list else "Napoleon Bonaparte leading his army across the Alps"

print(f"Testing with prompt: {test_prompt}")
print("=" * 60)

classification, justification, gen_time = classify_prompt(test_prompt)

print(f"Classification: {classification}")
print(f"Justification: {justification}")
print(f"Generation time: {gen_time:.2f}s")
print("=" * 60)

estimated_total_time = gen_time * len(prompts_list)
print(f"\nEstimated time for {len(prompts_list)} prompts: {estimated_total_time/60:.1f} minutes")

## 7. Run Sample Classification

In [ ]:
SAVE_EVERY = 100
RESULTS_FILE = "data/results_mlx/validation_results.csv"

# Check for existing results (for resuming)
start_idx = 0
existing_results = []

if os.path.exists(RESULTS_FILE):
    existing_df = pd.read_csv(RESULTS_FILE)
    start_idx = len(existing_df)
    existing_results = existing_df.to_dict('records')
    print(f"Resuming from index {start_idx} ({start_idx} prompts already processed)")

rows_to_process = prompts_df.iloc[start_idx:].reset_index(drop=True)
results = existing_results.copy()

print(f"Processing {len(rows_to_process)} prompts in batches of {BATCH_SIZE_INFERENCE}...")
print("=" * 60)

start_time = time.time()
total_batches = (len(rows_to_process) + BATCH_SIZE_INFERENCE - 1) // BATCH_SIZE_INFERENCE

for batch_idx in tqdm(range(total_batches), desc="Batch processing"):
    batch_start = batch_idx * BATCH_SIZE_INFERENCE
    batch_end = min(batch_start + BATCH_SIZE_INFERENCE, len(rows_to_process))
    batch_rows = rows_to_process.iloc[batch_start:batch_end]
    batch_prompts = batch_rows['prompt'].tolist()
    try:
        batch_results, batch_time = classify_batch(batch_prompts)
        for result, (_, row) in zip(batch_results, batch_rows.iterrows()):
            result['image_uri'] = row['image_uri']
            result['timestamp'] = row['timestamp']
            result['datetime'] = str(row['datetime']) if pd.notna(row['datetime']) else ''
            result['modifiers'] = row['modifiers']
            result['width'] = row['width']
            result['height'] = row['height']
        results.extend(batch_results)
    except Exception as e:
        logger.error(f"Error in batch {batch_idx}: {e}")
        for _, row in batch_rows.iterrows():
            try:
                classification, justification, gen_time = classify_prompt(row['prompt'])
                results.append({
                    'prompt': row['prompt'], 'references_past': classification,
                    'justification': justification, 'generation_time': gen_time,
                    'image_uri': row['image_uri'], 'timestamp': row['timestamp'],
                    'datetime': str(row['datetime']) if pd.notna(row['datetime']) else '',
                    'modifiers': row['modifiers'], 'width': row['width'], 'height': row['height'],
                })
            except Exception as e2:
                results.append({
                    'prompt': row['prompt'], 'references_past': 'error',
                    'justification': str(e2), 'generation_time': 0,
                    'image_uri': row['image_uri'], 'timestamp': row['timestamp'],
                    'datetime': str(row['datetime']) if pd.notna(row['datetime']) else '',
                    'modifiers': row['modifiers'], 'width': row['width'], 'height': row['height'],
                })
    if (batch_idx + 1) % (SAVE_EVERY // BATCH_SIZE_INFERENCE) == 0 or batch_idx == total_batches - 1:
        pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
        elapsed = time.time() - start_time
        prompts_done = len(results) - len(existing_results)
        rate = prompts_done / elapsed if elapsed > 0 else 0
        logger.info(f"Saved {len(results)} results. Rate: {rate:.1f} prompts/sec")

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_FILE, index=False)

total_time = time.time() - start_time
prompts_processed = len(results) - len(existing_results)
print("=" * 60)
print(f"Completed! Total time: {total_time/60:.1f} minutes")
print(f"Prompts processed: {prompts_processed}")
print(f"Average rate: {prompts_processed/total_time:.2f} prompts/sec")

## 8. Results Summary

In [ ]:
results_df = pd.read_csv(RESULTS_FILE)

print("=" * 60)
print("VALIDATION RESULTS SUMMARY")
print("=" * 60)

total = len(results_df)
yes_count = (results_df['references_past'] == 'yes').sum()
no_count = (results_df['references_past'] == 'no').sum()
error_count = (results_df['references_past'] == 'error').sum()

print(f"Total prompts processed: {total}")
print(f"References to past (yes): {yes_count} ({yes_count/total*100:.2f}%)")
print(f"No reference (no): {no_count} ({no_count/total*100:.2f}%)")
print(f"Errors: {error_count} ({error_count/total*100:.2f}%)")

avg_time = results_df['generation_time'].mean()
print(f"\nAverage generation time: {avg_time:.2f}s per prompt")
print(f"Throughput: {3600/avg_time:.0f} prompts/hour")

full_dataset_hours = (10_000_000 * avg_time) / 3600
print(f"\nEstimated time for 10M prompts: {full_dataset_hours:.0f} hours ({full_dataset_hours/24:.1f} days)")

In [ ]:
# Show sample results
print("\n" + "=" * 60)
print("SAMPLE RESULTS WITH JUSTIFICATIONS")
print("=" * 60)

yes_samples = results_df[results_df['references_past'] == 'yes'].head(5)
print("\n--- Prompts WITH references to the past ---")
for _, row in yes_samples.iterrows():
    print(f"\nPrompt: {row['prompt'][:100]}{'...' if len(row['prompt']) > 100 else ''}")
    print(f"Justification: {row['justification']}")

no_samples = results_df[results_df['references_past'] == 'no'].head(5)
print("\n--- Prompts WITHOUT references to the past ---")
for _, row in no_samples.iterrows():
    print(f"\nPrompt: {row['prompt'][:100]}{'...' if len(row['prompt']) > 100 else ''}")
    print(f"Justification: {row['justification']}")

## 9. Clustering with BERTopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import plotly.express as px

results_df = pd.read_csv("data/results_mlx/validation_results.csv")
historical_prompts = results_df[results_df['references_past'] == 'yes']['prompt'].tolist()

print(f"Clustering {len(historical_prompts)} prompts with historical references...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
vectorizer_model = CountVectorizer(stop_words='english', ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=2,
    verbose=True
)

topics, probs = topic_model.fit_transform(historical_prompts)

print(f"\nFound {len(set(topics)) - 1} topics (excluding outliers)")
print(f"Outliers (topic -1): {topics.count(-1)} prompts")

In [ ]:
# Topic info
print("=" * 60)
print("TOPICS DISCOVERED")
print("=" * 60)

topic_info = topic_model.get_topic_info()
print(topic_info[['Topic', 'Count', 'Name']].to_string())

print("\n" + "=" * 60)
print("REPRESENTATIVE PROMPTS PER TOPIC")
print("=" * 60)

for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    topic_prompts = [p for p, t in zip(historical_prompts, topics) if t == topic_id]
    topic_words = topic_model.get_topic(topic_id)
    print(f"\n--- Topic {topic_id} ({len(topic_prompts)} prompts) ---")
    print(f"Top words: {', '.join([w for w, _ in topic_words[:5]])}")
    print("Sample prompts:")
    for p in topic_prompts[:3]:
        print(f"  - {p[:80]}{'...' if len(p) > 80 else ''}")

### 9.1 Visualizations

In [ ]:
# Intertopic Distance Map
fig_topics = topic_model.visualize_topics()
fig_topics.show()
fig_topics.write_html("data/results_mlx/topics_map.html")
print("Saved: data/results_mlx/topics_map.html")

In [ ]:
# 2D scatter plot of prompts colored by topic
embeddings = embedding_model.encode(historical_prompts)

from umap import UMAP
umap_model = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embeddings_2d = umap_model.fit_transform(embeddings)

viz_df = pd.DataFrame({
    'x': embeddings_2d[:, 0], 'y': embeddings_2d[:, 1],
    'topic': [str(t) for t in topics],
    'prompt': [p[:100] + '...' if len(p) > 100 else p for p in historical_prompts]
})

fig = px.scatter(viz_df, x='x', y='y', color='topic', hover_data=['prompt'],
    title='Prompts with Historical References - Clustered by Semantic Similarity',
    width=900, height=700)
fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()
fig.write_html("data/results_mlx/prompts_clusters.html")
print("Saved: data/results_mlx/prompts_clusters.html")

### 9.2 Temporal Analysis

In [ ]:
import plotly.graph_objects as go

TEMPORAL_GRANULARITY = 'W'
GRANULARITY_LABELS = {'M': 'Monthly', 'W': 'Weekly', 'D': 'Daily'}

results_df = pd.read_csv("data/results_mlx/validation_results.csv")
results_df['datetime'] = pd.to_datetime(results_df['datetime'], errors='coerce')
valid_mask = results_df['datetime'].notna()
temporal_df = results_df[valid_mask].copy()

print(f"Prompts with valid timestamps: {len(temporal_df):,} / {len(results_df):,}")

temporal_df['period'] = temporal_df['datetime'].dt.to_period(TEMPORAL_GRANULARITY)
period_total = temporal_df.groupby('period').size()
period_historical = temporal_df[temporal_df['references_past'] == 'yes'].groupby('period').size()

temporal_stats = pd.DataFrame({
    'total': period_total, 'historical': period_historical,
}).fillna(0).astype(int)
temporal_stats['pct_historical'] = (temporal_stats['historical'] / temporal_stats['total'] * 100).round(2)

print(f"\n{'=' * 60}")
print(f"HISTORICAL REFERENCES OVER TIME ({GRANULARITY_LABELS.get(TEMPORAL_GRANULARITY)})")
print(f"{'=' * 60}")
for period, row in temporal_stats.iterrows():
    bar = '█' * int(row['pct_historical'])
    print(f"  {period}  {row['historical']:>4}/{row['total']:>5} ({row['pct_historical']:>5.1f}%) {bar}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[str(p) for p in temporal_stats.index], y=temporal_stats['pct_historical'],
    mode='lines+markers', name='% Historical',
    line=dict(color='coral', width=2), marker=dict(size=8),
    hovertemplate='%{x}<br>%{y:.1f}% historical<br>%{customdata[0]} / %{customdata[1]} prompts',
    customdata=temporal_stats[['historical', 'total']].values,
))
fig.update_layout(title=f'Proportion of Historical Prompts Over Time ({GRANULARITY_LABELS.get(TEMPORAL_GRANULARITY)})',
    xaxis_title='Period', yaxis_title='% Historical Prompts', width=900, height=400)
fig.show()
fig.write_html("data/results_mlx/temporal_historical_pct.html")
print("Saved: data/results_mlx/temporal_historical_pct.html")

In [ ]:
# Topic evolution over time - Streamgraph
import plotly.graph_objects as go

hist_mask = temporal_df['references_past'] == 'yes'
hist_temporal = temporal_df[hist_mask].copy()

prompt_to_topic = dict(zip(historical_prompts, topics))
hist_temporal['topic'] = hist_temporal['prompt'].map(prompt_to_topic)
hist_temporal = hist_temporal.dropna(subset=['topic'])
hist_temporal['topic'] = hist_temporal['topic'].astype(int)

topic_names = {}
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        topic_names[-1] = "Outliers"
    else:
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"T{topic_id}: {', '.join([w for w, _ in words[:3]])}"

hist_temporal['topic_name'] = hist_temporal['topic'].map(topic_names)

topic_period = hist_temporal.groupby(['period', 'topic_name']).size().reset_index(name='count')
topic_period['period_str'] = topic_period['period'].astype(str)
pivot = topic_period.pivot(index='period_str', columns='topic_name', values='count').fillna(0)

fig = go.Figure()
for col in pivot.columns:
    if col == "Outliers":
        continue
    fig.add_trace(go.Scatter(x=pivot.index, y=pivot[col], name=col, mode='lines', stackgroup='one',
        hovertemplate='%{x}<br>' + col + ': %{y} prompts'))
fig.update_layout(title=f'Topic Evolution Over Time ({GRANULARITY_LABELS.get(TEMPORAL_GRANULARITY)}) - Streamgraph',
    xaxis_title='Period', yaxis_title='Number of Prompts', width=1000, height=500,
    showlegend=True, legend=dict(font=dict(size=9)))
fig.show()
fig.write_html("data/results_mlx/topic_evolution_streamgraph.html")
print("Saved: data/results_mlx/topic_evolution_streamgraph.html")